In [ ]:
import numpy as np
import cv2
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# Embedding
class Embed(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=3):
        super().__init__()
        self.embed = nn.Conv2d(1, d, kernel_size=(p,p), stride=(p,p), padding=0)
        self.position_encoding = nn.Parameter(torch.zeros(h//p, w//p, d).float())
    def forward(self, x):
        x = self.embed(x) # B x 1 x h x w -> B x d x h//p x w//p
        x = x.permute(0, 2, 3, 1) # -> B x h//p x w//p x d
        x += self.position_encoding.unsqueeze(0)
        return x

In [ ]:
# Attention
class Attention(nn.Module):
    def __init__(self, d=3, num_heads=1):
        super(Attention, self).__init__()
        self.num_attention_heads = num_heads
        self.attention_head_size = d // self.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size # == d

        self.query = nn.Linear(d, self.all_head_size)
        self.key = nn.Linear(d, self.all_head_size)
        self.value = nn.Linear(d, self.all_head_size)

        self.out = nn.Linear(self.all_head_size, d)
        #self.attn_dropout = nn.Dropout(0)
        #self.proj_dropout = nn.Dropout(0)

        self.softmax = nn.Softmax(dim=-1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_probs = self.softmax(attention_scores)
        weights = attention_probs
        #attention_probs = self.attn_dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        attention_output = self.out(context_layer)
        #attention_output = self.proj_dropout(attention_output)
        return attention_output, weights

In [ ]:
def swish(x):
    return x * torch.sigmoid(x)

In [ ]:
# perceptron
class Mlp(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 4*d)
        self.fc2 = nn.Linear(4*d, d)
        self.act_fn = swish
        #self.dropout = nn.Dropout(0.1)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act_fn(x)
        #x = self.dropout(x)
        x = self.fc2(x)
        #x = self.dropout(x)
        return x

In [ ]:
# transformation
class Block(nn.Module):
    def __init__(self, n=50, d=3, num_heads=1):
        super().__init__()
        self.attention_norm = nn.LayerNorm(d)
        self.attn = Attention(d, num_heads)
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = Mlp(d)
    def forward(self, x):
        h = x
        x = self.attention_norm(x)
        x, weights = self.attn(x)
        x += h
        h = x
        x = self.ffn_norm(x)
        x = self.ffn(x)
        x += h
        return x, weights

In [ ]:
# wipeout
class Wipeout(nn.Module):
    def __init__(self, d=3, num_categories=10):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.wipeout = nn.Linear(d, num_categories, bias=False)
    def forward(self, x):
        return self.wipeout(self.norm(x)[:,0,:])

In [ ]:
# Transformer (encoder-only)
class Transformer(nn.Module):
    def __init__(self, h=28, w=28, p=4, d=3, num_heads=1, num_layers=6, num_categories=10):
        super().__init__()
        self.embed = Embed(h,w,p,d)
        self.cls = nn.Parameter(torch.zeros(1,1,d).float())
        self.pe = nn.Parameter(torch.zeros(h//p, w//p, d).float())
        self.encoder = nn.ModuleList()
        for _ in range(num_layers):
            self.encoder.append(Block((h//p)*(w//p)+1, d, num_heads))
        self.wipeout = Wipeout(d,num_categories)
    def forward(self, x):
        hidden_states = self.embed(x) + self.pe # -> B x (h/p) x (w/p) x d
        B, H, W, d = hidden_states.shape
        hidden_states = hidden_states.view(B,H*W,d) # -> B x (h/p)*(w/p) x d
        cls_tokens = self.cls.expand(B, -1, -1)  # B x 1 x d
        hidden_states = torch.cat([cls_tokens, hidden_states], dim=1) # -> B x (h/p)*(w/p)+1 x d
        attn_weights = []
        intermediates = [hidden_states]
        for block in self.encoder:
            hidden_states, weights = block(hidden_states)
            attn_weights.append(weights)
            intermediates.append(hidden_states)
        return self.wipeout(hidden_states), torch.stack(attn_weights).permute(1,0,2,3,4), torch.stack(intermediates).permute(1,0,2,3)
        # B x l, B x num_layers x num_heads x (h/p)*(w/p)+1 x (h/p)*(w/p)+1 x, B x num_layers+1 x (h/p)*(w/p)+1 x d

In [ ]:
# create classifier
model = Transformer(28,28,4,3,1,6,3).to(device) # height, width, patch size, dimension, number of heads, number of blocks, number of categories

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
# select a subset of classes
classes_to_keep = [0, 6, 9]
class_to_id = {class_: i for i, class_ in enumerate(classes_to_keep)}
print(class_to_id)
id_to_class = {i: class_ for class_, i in class_to_id.items()}
print(id_to_class)

In [ ]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [ ]:
# create subset
train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in classes_to_keep] # indices for train subset
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in classes_to_keep] # indices for test subset
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

In [ ]:
# create train and test dataloader
batch_size = 512
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [ ]:
# Define loss funcion
criterion = nn.CrossEntropyLoss()

In [ ]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# scheduler
step_size = 8
gamma = 0.8 # 0.75
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# training
history_loss = []
history_acc = []
history_test_acc = []
epoch = 0

In [ ]:
def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # change model in training mood
        model.train()

        # to record loss and accuracy
        batch_loss = []
        total_train = 0
        correct_train = 0

        for batch, (x_train, y_train) in enumerate(train_loader):

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output, _, _ = model(input)

            # categorization
            expected_output = y_train.to(device)

            # cross entropy loss
            loss = criterion(output, expected_output)

            # find gradients
            loss.backward()
            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss.append(loss.item())

            # recording accuracy
            total_train += output.shape[0]
            correct_train += torch.argmax(output,dim=1).to('cpu').eq(y_train).sum().item()

        epoch_loss = np.average(batch_loss)
        epoch_acc = (100.0 * correct_train) / (total_train+1e-7)

        history_loss.append(epoch_loss)
        history_acc.append(epoch_acc)

        total_test = 0
        correct_test = 0

        model.eval()

        for batch, (x_test, y_test) in enumerate(test_loader):

            # send data to device
            input = x_test.to(device)

            # forward pass to the model
            with torch.no_grad():
                output, _, _ = model(input)

            # recording accuracy
            total_test += output.shape[0]
            correct_test += torch.argmax(output,dim=1).to('cpu').eq(y_test).sum().item()

        test_acc = (100.0 * correct_test) / (total_test+1e-7)

        history_test_acc.append(test_acc)

        print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Accuracy: {epoch_acc:.4f} Test accuracy: {test_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
# training
num_epochs = 120
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
train(num_epochs)

In [ ]:
def plot_history():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # 1 row, 2 columns
    # Loss curve
    axes[0].plot(history_loss, 'r', linewidth=3.0)
    axes[0].set_title('Loss Curve', fontsize=16)
    axes[0].set_xlabel('Epochs', fontsize=14)
    axes[0].set_ylabel('Loss', fontsize=14)
    axes[0].legend(['Training Loss'], fontsize=12)
    # Accuracy curves
    axes[1].plot(history_acc, 'r', linewidth=3.0)
    axes[1].plot(history_test_acc, 'b', linewidth=3.0)
    axes[1].set_title('Accuracy Curves', fontsize=16)
    axes[1].set_xlabel('Epochs', fontsize=14)
    axes[1].set_ylabel('Accuracy', fontsize=14)
    axes[1].legend(['Training Accuracy', 'Validation Accuracy'], fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_history()

In [ ]:
# Save model and weights
def save():
    torch.save(model.state_dict(), classifier_name) # weights only
    print('Saved trained model at %s ' % classifier_name)

In [ ]:
classifier_name = 'mnist_ViT3.pth'
save()

In [ ]:
# download model
from google.colab import files
files.download(classifier_name)

In [ ]:
# instead of trainig, we can download its result
#!wget http://agentspace.org/download/mnist_ViT3.pth
#model.load_state_dict(torch.load("mnist_ViT3.pth", map_location=torch.device(device)))

In [ ]:
# get few samples
test_iter = iter(test_loader)
x_sample, y_sample = next(test_iter)
print(x_sample.shape)

In [ ]:
# use the model on the samples
input_images = x_sample[0:100].to(device)
output_probabilities, _, _ = model(input_images)
output_categories = [ id_to_class[category.item()] for category in torch.argmax(output_probabilities.to('cpu'), dim=1) ]
print(output_categories)
categories = [ id_to_class[category.item()] for category in y_sample ]
print(categories)

In [ ]:
# show results
def render(digit):
    img = np.zeros((28, 28), dtype=np.uint8)
    cv2.putText(img, str(digit), (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    return img

plt.figure(figsize=(20, 40))
for b in range(10):
    for i in range(10):
        input_img = (x_sample[10*b+i].squeeze(0).detach().numpy()*255).astype(np.uint8)
        plt.subplot(20, 10, 20*b+i+1)
        plt.imshow(input_img, cmap='gray')
        plt.axis('off')
        output_img = render(output_categories[10*b+i])
        plt.subplot(20, 10, 20*b+i+1+10)
        plt.imshow(~output_img, cmap='gray')
        plt.axis('off')

plt.show()

In [ ]:
# collect hidden states for all samples from the testing dataset
hidden_states = []
categories = []
with torch.no_grad():
    for samples_x, samples_y in test_loader:
        _, _, intermediates = model(samples_x.to(device))
        hidden_states.append(intermediates)
        categories.append(samples_y)

In [ ]:
hidden_states = torch.cat(hidden_states,dim=0)
categories = torch.cat(categories,dim=0)

In [ ]:
hidden_states.shape

In [ ]:
# visualize latent space after several transformations
def visualize(i=0):
    points = hidden_states[:,i,0,:].detach().cpu().numpy()
    colors = px.colors.qualitative.T10
    traces = []
    for i in range(3):
        mask = categories == i
        pts = points[mask]
        scatter = go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers', marker=dict(size=5, color=colors[i]), name=id_to_class[i])
        traces.append(scatter)
    fig = go.Figure(data=traces)
    fig.update_layout(
        scene=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z')),
        legend=dict(x=1.05, y=1),
        margin=dict(l=0, r=200, t=50, b=0)
    )
    fig.show()

In [ ]:
visualize(0)

In [ ]:
visualize(1)

In [ ]:
visualize(2)

In [ ]:
visualize(3)

In [ ]:
visualize(4)

In [ ]:
visualize(5)

In [ ]:
visualize(6)

In [ ]:
x_sample[0].unsqueeze(0).shape

In [ ]:
sample = x_sample[0]
with torch.no_grad():
     _, attmaps, _ = model(sample.unsqueeze(0).to(device))

In [ ]:
attmaps.shape

In [ ]:
attmaps_ = attmaps[0,:,0,0,1:].detach().cpu().numpy().reshape(-1,7,7)

In [ ]:
attmaps_.shape

In [ ]:
fig = plt.figure(figsize=(14, 4))
ax_main = plt.subplot2grid((2, 8), (0, 0), rowspan=2, colspan=2)  # main image
ax_main.imshow(sample.squeeze(0).detach().cpu().numpy(), cmap='gray')
ax_main.axis('off')

for i, attmap in enumerate(attmaps_):
    ax = plt.subplot2grid((2, 8), (0, i+2), rowspan=2, colspan=1)
    ax.imshow(attmap, cmap='gray')
    ax.axis('off')
    ax.set_title(f'{i+1}')

plt.tight_layout()
plt.show()

In [ ]:
att_mats = attmaps[0]
print(att_mats.shape)
num_layers, num_heads, grid_size, _ = att_mats.shape

In [ ]:
# joined attention
att = att_mats.mean(dim=1)
joint_att = att[0]
for l in range(1, num_layers):
    joint_att = att[l] @ joint_att

In [ ]:
joint_att.shape

In [ ]:
plt.imshow(joint_att[0,1:].view(7,7).detach().cpu().numpy(), cmap='gray')
plt.show()